# Reproduction: Pipeline

This jupyter notebook illustrates how to reproduce the model training (and evaluation) process of the linear regression model and the ridge regression model. Besides, all code parts used in this notebook can be found in the project repo.

# 1. Environment Setup

In [ ]:
# Install dependencies if not already present
!pip install pm4py scikit-learn joblib matplotlib pandas

# 2. Configure System Path

In [ ]:
import sys
import os

src_path = os.path.abspath(os.path.join('..', 'src'))

if src_path not in sys.path:
    sys.path.append(src_path)
    print(f"Added {src_path} to sys.path")

os.chdir(os.path.abspath(os.path.join('..')))
print(f"Current Working Directory: {os.getcwd()}")

# 3. Data Acquisition

We run the setup script to download and extract the BPI Challenge 2013 dataset.

In [ ]:
import src.data_setup as data_setup

print("--- Step 1: Downloading and Extracting Data ---")
data_setup.setup_environment()
if data_setup.download_data():
    data_setup.extract_data()

# 4. Run the Integrated Pipeline

Import the main function from your source code. This will run the preprocessing, all models, and generate the visualizations.

In [ ]:
from src.main import main

# Execute the full pipeline logic
# Note: Since we changed the working directory to root, config.py will find the folders correctly.
main()

# 5. Review Evaluation Scores

Inspect the model_scores.csv to see how the OLS and Ridge models compare to the baseline.

In [ ]:
import pandas as pd
from src.config import ARTIFACTS_DIR

scores_path = os.path.join(ARTIFACTS_DIR, "model_scores.csv")
df_scores = pd.read_csv(scores_path)

print("--- Model Comparison Summary ---")
display(df_scores)

# 6. Review Detailed Metrics

Check the MAE, RMSE, MedAE, and R2 for specific prefix lengths

In [ ]:
metrics_path = os.path.join(ARTIFACTS_DIR, "model_metrics.csv")
df_metrics = pd.read_csv(metrics_path)

print("--- Detailed Metrics (First 10 Rows) ---")
display(df_metrics.head(10))

# 7. Display Visualizations

Since the main() function already generated the .png files, we can just load them here.

In [ ]:
from IPython.display import Image, display as display_img
from src.config import FIGURES_DIR

print("--- Comparison Plots ---")
for metric in ["mae", "rmse", "medae", "r2"]:
    path = os.path.join(FIGURES_DIR, f"compare_{metric}.png")
    if os.path.exists(path):
        print(f"Metric: {metric.upper()}")
        display_img(Image(filename=path))

print("\n--- Model Profile Plots ---")
for profile in ["ols_pipeline", "pipe_ridge"]:
    path = os.path.join(FIGURES_DIR, f"profile_{profile}.png")
    if os.path.exists(path):
        print(f"Model: {profile}")
        display_img(Image(filename=path))

# 8. Test a Prediction

Show how to use the saved joblib model for real-time inference.

In [ ]:
from joblib import load
from src.pipeline_helper import preprocess_data

# Load the saved Ridge pipeline
model_path = os.path.join(ARTIFACTS_DIR, 'pipe_ridge.pkl')
trained_pipe = load(model_path)

# Load data using your existing helper
_, _, test_log = preprocess_data()

# Take 5 events from the test set
sample_data = test_log.head(5)
predictions = trained_pipe.predict(sample_data)

print("Input Events (Prefixes):")
display(sample_data[['case:concept:name', 'concept:name', 'event_count']])
print("\nPredicted Remaining Time (Hours):")
print(predictions)